In [1]:
import os

# Path to your dataset folder
dataset_path = "/content/drive/MyDrive/Breast Cancer/Dataset_BUSI/Dataset_BUSI_with_GT"

# Class names in your dataset
classes = ["normal", "benign", "malignant"]

for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    image_count = 0
    mask_count = 0

    for file in os.listdir(cls_path):
        if file.endswith((".png")):
            if "mask" in file.lower():
                mask_count += 1
            else:
                image_count += 1

    print(f"Class '{cls}': {image_count} images, {mask_count} masks")


Class 'normal': 133 images, 133 masks
Class 'benign': 437 images, 454 masks
Class 'malignant': 210 images, 211 masks


In [6]:
import os

# Path to your dataset folder
dataset_path =  "/content/drive/MyDrive/Breast Cancer/Dataset_BUSI/Dataset_BUSI_with_GT"
classes = ["normal", "benign", "malignant"]

for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    images = []
    masks = []

    # Collect image and mask filenames
    for file in os.listdir(cls_path):
        if file.endswith((".png")):
            name = os.path.splitext(file)[0]
            if "mask" in name.lower():
                masks.append(name.replace("_mask",""))
            else:
                images.append(name)

    # Find valid pairs
    valid_pairs = set(images).intersection(set(masks))
    unmatched_images = set(images) - valid_pairs
    unmatched_masks = set(masks) - valid_pairs

    print(f"\nClass '{cls}':")
    print(f"  Valid pairs: {len(valid_pairs)}")
    print(f"  Unmatched images: {len(unmatched_images)} -> {unmatched_images}")
    print(f"  Unmatched masks: {len(unmatched_masks)} -> {unmatched_masks}")



Class 'normal':
  Valid pairs: 133
  Unmatched images: 0 -> set()
  Unmatched masks: 0 -> set()

Class 'benign':
  Valid pairs: 437
  Unmatched images: 0 -> set()
  Unmatched masks: 17 -> {'benign (163)_1', 'benign (195)_2', 'benign (100)_1', 'benign (92)_1', 'benign (54)_1', 'benign (25)_1', 'benign (58)_1', 'benign (4)_1', 'benign (98)_1', 'benign (195)_1', 'benign (173)_1', 'benign (181)_1', 'benign (315)_1', 'benign (83)_1', 'benign (93)_1', 'benign (346)_1', 'benign (424)_1'}

Class 'malignant':
  Valid pairs: 210
  Unmatched images: 0 -> set()
  Unmatched masks: 1 -> {'malignant (53)_1'}


In [9]:
import os
import cv2
import numpy as np

dataset_path = "/content/drive/MyDrive/Breast Cancer/Dataset_BUSI/Dataset_BUSI_with_GT"
classes = ["normal", "benign", "malignant"]

for cls in classes:
    cls_path = os.path.join(dataset_path, cls)

    # Get all image files (ignore masks)
    images = [f for f in os.listdir(cls_path) if f.endswith(".png") and "_mask" not in f]

    for img_file in images:
        img_name = os.path.splitext(img_file)[0]

        # Find all mask files for this image
        mask_files = [f for f in os.listdir(cls_path) if f.startswith(img_name) and "_mask" in f]

        if len(mask_files) <= 1:
            continue  # nothing to merge

        combined_mask = None

        for mask_file in mask_files:
            mask_path = os.path.join(cls_path, mask_file)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = (mask > 0).astype(np.uint8)  # binarize mask
            if combined_mask is None:
                combined_mask = mask
            else:
                combined_mask = np.maximum(combined_mask, mask)  # merge masks

        # Save the combined mask (overwrite first mask)
        save_path = os.path.join(cls_path, img_name + "_mask.png")
        cv2.imwrite(save_path, combined_mask * 255)

        # Optionally delete extra masks
        for mask_file in mask_files:
            if mask_file != img_name + "_mask.png":
                os.remove(os.path.join(cls_path, mask_file))

        print(f"Merged {len(mask_files)} masks for {img_file}")


Merged 2 masks for benign (100).png
Merged 2 masks for benign (173).png
Merged 2 masks for benign (181).png
Merged 2 masks for benign (25).png
Merged 2 masks for benign (346).png
Merged 2 masks for benign (315).png
Merged 2 masks for benign (58).png
Merged 2 masks for benign (4).png
Merged 2 masks for benign (54).png
Merged 2 masks for benign (424).png
Merged 2 masks for benign (93).png
Merged 2 masks for benign (92).png
Merged 2 masks for benign (98).png
Merged 2 masks for benign (83).png
Merged 3 masks for benign (195).png
Merged 2 masks for benign (163).png
Merged 2 masks for malignant (53).png


In [10]:
import os

# Path to your dataset folder
dataset_path = "/content/drive/MyDrive/Breast Cancer/Dataset_BUSI/Dataset_BUSI_with_GT"

# Class names in your dataset
classes = ["normal", "benign", "malignant"]

for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    image_count = 0
    mask_count = 0

    for file in os.listdir(cls_path):
        if file.endswith((".png")):
            if "mask" in file.lower():
                mask_count += 1
            else:
                image_count += 1

    print(f"Class '{cls}': {image_count} images, {mask_count} masks")


Class 'normal': 133 images, 133 masks
Class 'benign': 437 images, 437 masks
Class 'malignant': 210 images, 210 masks


In [5]:
import tensorflow as tf
print(tf.__version__)

2.19.0


In [6]:
import os
import pandas as pd
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# -------------------
# CONFIG
# -------------------
ROOT = "/content/drive/MyDrive/Breast Cancer/Dataset_BUSI/Dataset_BUSI_with_GT"
CLASSES = ["normal", "benign", "malignant"]
IMG_SIZE = 256
OUTPUT_DIR = "/content/drive/MyDrive/Breast Cancer/Processed"

# -------------------
# STEP 1: Build dataframe with image & mask paths
# -------------------
data = []

for cls in CLASSES:
    cls_path = os.path.join(ROOT, cls)
    for f in os.listdir(cls_path):
        if f.endswith(".png") and "_mask" not in f:
            img_path = os.path.join(cls_path, f)
            mask_path = os.path.join(cls_path, f.replace(".png", "_mask.png"))
            if os.path.exists(mask_path):
                data.append([img_path, mask_path, cls])

df = pd.DataFrame(data, columns=["image", "mask", "label"])
print("Total images:", len(df))
print(df.head())

# -------------------
# STEP 2: Split dataset (train 70%, val 15%, test 15%)
# -------------------
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# -------------------
# STEP 3: Preprocessing functions using OpenCV
# -------------------
def preprocess_image(img_path):
    # Read image in color
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # Median filter (speckle noise removal)
    img = cv2.medianBlur(img, 3)

    # CLAHE (Contrast Limited Adaptive Histogram Equalization)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    # Gaussian smoothing
    img = cv2.GaussianBlur(img, (3,3), 1.0)

    # Normalize to 0-1
    img = img.astype(np.float32) / 255.0
    return img

def preprocess_mask(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = (mask > 0).astype(np.uint8)  # binary mask
    return mask

# -------------------
# STEP 4: Setup output directories
# -------------------
def create_dirs(base_path, classes):
    for split in ['train','val','test']:
        for cls in classes:
            os.makedirs(os.path.join(base_path, split, cls, 'images'), exist_ok=True)
            os.makedirs(os.path.join(base_path, split, cls, 'masks'), exist_ok=True)

create_dirs(OUTPUT_DIR, CLASSES)

# -------------------
# STEP 5: Save images
# -------------------
def save_image_mask(img, mask, save_img_path, save_mask_path):
    # Convert back to uint8
    img_uint8 = (img * 255).astype(np.uint8)
    mask_uint8 = (mask * 255).astype(np.uint8)

    cv2.imwrite(save_img_path, img_uint8)
    cv2.imwrite(save_mask_path, mask_uint8)

# -------------------
# STEP 6: Preprocess and save a split
# -------------------
def process_and_save(df_split, split_name, base_path):
    for idx, row in tqdm(df_split.iterrows(), total=len(df_split), desc=f"Processing {split_name}"):
        img = preprocess_image(row['image'])
        mask = preprocess_mask(row['mask'])

        cls = row['label']
        img_name = os.path.basename(row['image'])
        mask_name = os.path.basename(row['mask'])

        save_img_path = os.path.join(base_path, split_name, cls, 'images', img_name)
        save_mask_path = os.path.join(base_path, split_name, cls, 'masks', mask_name)

        save_image_mask(img, mask, save_img_path, save_mask_path)

# -------------------
# STEP 7: Apply to train/val/test
# -------------------
process_and_save(train_df, 'train', OUTPUT_DIR)
process_and_save(val_df, 'val', OUTPUT_DIR)
process_and_save(test_df, 'test', OUTPUT_DIR)

print("Preprocessing and saving completed!")


Total images: 780
                                               image  \
0  /content/drive/MyDrive/Breast Cancer/Dataset_B...   
1  /content/drive/MyDrive/Breast Cancer/Dataset_B...   
2  /content/drive/MyDrive/Breast Cancer/Dataset_B...   
3  /content/drive/MyDrive/Breast Cancer/Dataset_B...   
4  /content/drive/MyDrive/Breast Cancer/Dataset_B...   

                                                mask   label  
0  /content/drive/MyDrive/Breast Cancer/Dataset_B...  normal  
1  /content/drive/MyDrive/Breast Cancer/Dataset_B...  normal  
2  /content/drive/MyDrive/Breast Cancer/Dataset_B...  normal  
3  /content/drive/MyDrive/Breast Cancer/Dataset_B...  normal  
4  /content/drive/MyDrive/Breast Cancer/Dataset_B...  normal  
Train: 546 Val: 117 Test: 117


Processing test: 100%|██████████| 117/117 [00:41<00:00,  2.80it/s]

Preprocessing and saving completed!
